# BM25 Retriever 성능 분석 (Hit@k & MRR)

이 노트북은 **BM25Retriever**의 키워드 검색 성능을 `golden_dataset.json`을 기반으로 정량적으로 평가합니다.

In [6]:
# ✅ 실험 파라미터 설정
SAMPLE_SIZE = 50   # 테스트할 샘플 개수
K_VALUES = [1, 3, 5]  # Hit@k 측정 기준

# 환경 설정
import sys
import os
from dotenv import load_dotenv
import pandas as pd
import json
import random

# 프로젝트 루트 경로 설정
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(project_root)

load_dotenv(os.path.join(project_root, ".env"))
print(f"Project Root: {project_root}")

Project Root: /Users/leemdo/Workspaces/SKN22-3rd-3Team


In [7]:
from src.retrieval.bm25_retriever import BM25Retriever

retriever = BM25Retriever(version="v3", collection_name="care_guides")
print("✅ BM25Retriever initialized.")

✅ BM25Retriever initialized.


## 1. 데이터셋 로드

In [8]:
dataset_path = os.path.join(project_root, "data/v3/golden_dataset.json")

if os.path.exists(dataset_path):
    with open(dataset_path, "r", encoding="utf-8") as f:
        FULL_DATASET = json.load(f)
    print(f"📚 Loaded {len(FULL_DATASET)} test cases.")
else:
    print("⚠️ Dataset not found. Using mock.")
    FULL_DATASET = [{"query": "메인쿤 특징", "expected_keyword": "메인 쿤"}]

# 샘플링
test_dataset = random.sample(FULL_DATASET, min(SAMPLE_SIZE, len(FULL_DATASET)))
print(f"🧪 Selected {len(test_dataset)} samples.")

📚 Loaded 3354 test cases.
🧪 Selected 50 samples.


## 2. 평가 로직 구현 (Hit@k, MRR)

In [9]:
async def evaluate_bm25(dataset, k_values):
    results_summary = {k: 0 for k in k_values}
    total_mrr = 0
    log_data = []
    
    max_k = max(k_values)
    
    for item in dataset:
        query = item.get("query")
        target = item.get("expected_keyword", item.get("target"))
        
        if not target: continue
        
        # 검색
        specialist = item.get("specialist")
        search_results = await retriever.search(query, specialist=specialist, limit=max_k)
        
        # 정답 확인
        found_rank = None
        for rank, res in enumerate(search_results):
            content = (
                res.get('name_ko', '') + " " + 
                res.get('title_refined', '') + " " + 
                res.get('text', '')
            ).lower()
            
            if target.lower() in content:
                found_rank = rank + 1
                break
        
        # Metrics 계산
        reciprocal_rank = 0
        hit_status = "❌"
        
        if found_rank:
            reciprocal_rank = 1 / found_rank
            hit_status = f"✅ (Rank {found_rank})"
            
            for k in k_values:
                if found_rank <= k:
                    results_summary[k] += 1
        
        total_mrr += reciprocal_rank
        
        log_data.append({
            "Query": query,
            "Target": target,
            "Result": hit_status,
            "MRR": round(reciprocal_rank, 4)
        })
        
    # 최종 집계
    total_count = len(dataset)
    metrics = {
        f"Hit@{k}": round(count / total_count, 4) 
        for k, count in results_summary.items()
    }
    metrics["MRR"] = round(total_mrr / total_count, 4)
    
    return metrics, pd.DataFrame(log_data)

print("Evaluator ready.")

Evaluator ready.


In [10]:
metrics, df_log = await evaluate_bm25(test_dataset, K_VALUES)

print("="*40)
print("   📊 BM25 Evaluation Results")
print("="*40)
for k, v in metrics.items():
    print(f"{k:<10}: {v}")
print("="*40)

display(df_log.head(10))

🔍 [BM25 RETRIEVER]: '고양이를 기죽게 만드는 행동은?' 검색 중 (전문가: Peacekeeper, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '대체할 수 있는 안전한 음식은 무엇인가요?' 검색 중 (전문가: Physician, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '고양이와 소통하는 방법은?' 검색 중 (전문가: Liaison, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '고양이가 쥐를 사냥하는 이유는?' 검색 중 (전문가: Liaison, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '고양이와의 소통 방법은?' 검색 중 (전문가: Peacekeeper, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '고양이 건강을 위한 병원 기준은?' 검색 중 (전문가: Liaison, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '고양이 성격에 맞는 별명은 무엇인가요?' 검색 중 (전문가: Liaison, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '고양이 화장실 청소의 중요성은?' 검색 중 (전문가: Peacekeeper, 필터: None)...
✅ [BM25 RETRIEVER]: 5건의 결과를 찾았습니다.
🔍 [BM25 RETRIEVER]: '고양이 새우 간식 추천은?' 검색 중 (전문가: General, 필터: None)...
✅ [BM25 RETRIEVER]: 4건의 결과를 찾았습니다.
🔍 [BM25 RETRI

,Query,Target,Result,MRR
0,고양이를 기죽게 만드는 행동은?,고양이를 기죽게 하는 집사의 행동 4가지,✅ (Rank 1),1.0000
1,대체할 수 있는 안전한 음식은 무엇인가요?,고양이 소시지 섭취의 위험성,✅ (Rank 5),0.2000
2,고양이와 소통하는 방법은?,고양이 언어와 바디랭귀지 이해하기,✅ (Rank 3),0.3333
3,고양이가 쥐를 사냥하는 이유는?,고양이 쥐 장난감 추천 10가지,✅ (Rank 1),1.0000
4,고양이와의 소통 방법은?,고양이 표정의 의미,❌,0.0000
5,고양이 건강을 위한 병원 기준은?,고양이 친화 병원 인증의 중요성,✅ (Rank 1),1.0000
6,고양이 성격에 맞는 별명은 무엇인가요?,고양이 성격에 맞는 별명 추천,✅ (Rank 1),1.0000
7,고양이 화장실 청소의 중요성은?,고양이 화장실 청소 시 감시하는 이유,✅ (Rank 1),1.0000
8,고양이 새우 간식 추천은?,고양이 새우 간식 추천 11가지,❌,0.0000
9,고양이 수염이 빠지는 이유는 무엇인가요?,고양이 수염의 비밀과 행운의 상징,✅ (Rank 2),0.5000
